# Module 6: Dynamic Swarm

Apply **Pattern 4**: agents hand off autonomously — no fixed path, no orchestrator. The route through the team **emerges at runtime** based on what each agent decides to do next.

![Dynamic Swarm: Monitor, Network Specialist, DB Admin, Resolver — autonomous handoffs, no fixed routing](./architecture.png)

**When to use this pattern:**
- Path cannot be known in advance
- Benefits from diverse specialist perspectives
- Exploration, brainstorming, incident response

**Key Strands primitive:** `Swarm([agents], entry_point=agent, max_handoffs=..., max_iterations=...)`

The `description` field on each agent is the routing signal: peers read it to decide who to hand off to.

**Prerequisites:** Modules 1–5.

## Agents in This Module

| Agent | Description (used for routing) |
|-------|-------------------------------|
| `monitor` | **Entry point.** Detects symptoms, assesses impact, classifies the incident type, decides which specialist to involve first. |
| `network_specialist` | Investigates network and infrastructure issues — load balancer, CDN, DNS, connectivity. |
| `db_admin` | Investigates database issues — slow queries, connection pool exhaustion, locks, index problems. |
| `resolver` | Synthesizes all findings and produces the resolution plan. **Final agent — does not hand off.** |

> **Why the path is dynamic:** Depending on the incident, the route could be `monitor → network_specialist → resolver`, `monitor → db_admin → resolver`, or `monitor → network_specialist → db_admin → resolver`. No code programs this — agents decide based on their `description`.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4, Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")
print("✅ Setup complete!")

In [ ]:
from strands import Agent
from strands.multiagent import Swarm

---

## Part 1: Create Swarm Agents

Each agent has a `description`: this is the routing signal. When an agent needs to hand off, it reads the descriptions of all peers and decides who is best suited for the next step.

The `system_prompt` tells each agent what to do when it receives a task. The last agent (`writer`) is told **not** to hand off: it produces the final output.

In [ ]:
INCIDENT_REPORT = '''
INCIDENT REPORT: E-Commerce Checkout Degradation

Time detected: 03:42 UTC — 15 minutes after deployment v2.4.1
Symptoms:
  - Checkout API latency: 8,200ms (baseline: 450ms)
  - Error rate: 12% (HTTP 5xx on /api/checkout)
  - Database connection pool: 95% utilized (baseline: 40%)
Impact: ~320 failed checkouts/min | Revenue loss: ~$4.2k/min
Affected services: checkout-service, order-api

Investigate root cause and produce a resolution plan.
'''

monitor = Agent(
    name="monitor",
    description=(
        "Entry point for all incidents. Detects symptoms, assesses impact, and classifies "
        "the incident type (network, database, or mixed). Routes to the right specialist first."
    ),
    system_prompt=(
        "You are a site reliability monitor. Analyze the incident report: identify symptoms, "
        "assess severity, classify whether this looks like a network/infra issue, a database issue, "
        "or both. Then hand off to the appropriate specialist (network_specialist or db_admin)."
    ),
    callback_handler=None,
)

network_specialist = Agent(
    name="network_specialist",
    description=(
        "Investigates network and infrastructure root causes — load balancer health, CDN, "
        "DNS resolution, inter-service connectivity, TLS/cert issues. Consult me when the "
        "incident might be caused by network or infrastructure problems."
    ),
    system_prompt=(
        "You are a network specialist. Investigate the incident from a network and infrastructure "
        "angle: load balancer config, CDN cache, DNS, service mesh. Report your findings. "
        "If database involvement is suspected, hand off to db_admin. "
        "If investigation is complete, hand off to resolver."
    ),
    callback_handler=None,
)

db_admin = Agent(
    name="db_admin",
    description=(
        "Investigates database root causes — slow queries, connection pool exhaustion, "
        "lock contention, index degradation, schema migration side effects. Consult me when "
        "DB metrics (connection pool, query latency, lock waits) are abnormal."
    ),
    system_prompt=(
        "You are a database administrator. Investigate the incident from a database angle: "
        "connection pool usage, slow query patterns, locks, recent schema changes from the deployment. "
        "Report your findings. Hand off to resolver when done."
    ),
    callback_handler=None,
)

resolver = Agent(
    name="resolver",
    description=(
        "Synthesizes all specialist findings and produces the incident resolution plan with "
        "immediate actions, root cause summary, and prevention steps. Use me last."
    ),
    system_prompt=(
        "You are an incident resolver. Synthesize all findings from the team into a resolution plan:\n"
        "## Root Cause\n"
        "## Immediate Actions (ordered by priority)\n"
        "## Verification Steps\n"
        "## Prevention (what to change before next deployment)\n"
        "This is the FINAL step. Do NOT hand off to anyone else."
    ),
    callback_handler=None,
)

---

## Part 2: Create and Run the Swarm

Unlike `GraphBuilder`, the Swarm has **no edges**: no fixed route is programmed. The Strands SDK automatically gives each agent a `handoff_to_agent` tool. Agents use it autonomously.

In [ ]:
import time

swarm = Swarm(
    [monitor, network_specialist, db_admin, resolver],
    entry_point=monitor,        # monitor always starts the investigation
    max_handoffs=8,             # safety: 4 agents, up to 2 passes each
    max_iterations=12,
    execution_timeout=300.0,
    node_timeout=180.0,         # 180s per node — LLM calls on cold containers can take 90-120s
)

print("Running Swarm (no fixed path — agents route autonomously based on findings)...")
t0 = time.time()
result = swarm(INCIDENT_REPORT)
total_time = time.time() - t0

print(f"\nStatus: {result.status} | {total_time:.1f}s")
print(f"Path emerged: {' → '.join(n.node_id for n in result.node_history)}")

---

## Part 3: Inspect the Emerged Path and Final Output

In [ ]:
print("=== SWARM PATH ===")
for i, node in enumerate(result.node_history):
    print(f"  {i+1}. {node.node_id}")

print()
print("=== RESOLUTION PLAN (resolver output) ===")
print("-" * 60)
print(str(result.results.get("resolver", "No resolver output")))
print("-" * 60)

In [ ]:
# Token usage across the full swarm
usage = result.accumulated_usage
print(f"{'Metric':<20} {'Value':>10}")
print("-" * 32)
print(f"{'Input tokens':<20} {usage.get('inputTokens', 0):>10}")
print(f"{'Output tokens':<20} {usage.get('outputTokens', 0):>10}")
print(f"{'Total tokens':<20} {usage.get('totalTokens', 0):>10}")
print(f"{'Agents in path':<20} {len(result.node_history):>10}")
print(f"{'Execution time':<20} {total_time:>9.1f}s")

---

## Part 4: Why the Path is Dynamic

The route that emerges depends on the incident. The same swarm with a different input might take a different path:

| Incident type | Likely path |
|---------------|-------------|
| Network/CDN outage | `monitor → network_specialist → resolver` |
| DB connection exhaustion | `monitor → db_admin → resolver` |
| Mixed (deployment + DB) | `monitor → db_admin → network_specialist → resolver` |
| Unknown root cause | `monitor → network_specialist → db_admin → resolver` |

No code programs which path to take. The `description` fields are the only routing signal.

| | Sequential Chain (M3) | Dynamic Swarm (M6) |
|---|---|---|
| Path | Fixed in Python code | Emergent: agents decide |
| Control | Deterministic | Autonomous |
| Change path | Rewrite code | Change `description` |
| Best for | Known, stable workflows | Exploration, incident response |

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `Swarm([agents], entry_point=agent)` | No routing code — agents decide the path |
| `description` field | Routing signal: write it for the model, not for humans |
| `handoff_to_agent` | Automatically added to each agent by the SDK |
| `result.node_history` | The path that emerged at runtime |
| `result.results["agent_name"]` | Output from a specific agent |
| `result.accumulated_usage` | Total token usage across all agents |

---

## What's Next

In **Module 7: Agent-as-Tool**, specialists are wrapped as tools and called by an orchestrator Agent with an LLM — combining explicit orchestration with specialist depth.